In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Embedding, LSTM, Dense, AdditiveAttention, Concatenate
)

In [2]:
# 1. Training data
pairs = [
    ("hello", "bonjour"),
    ("good morning", "bonjour"),
    ("thank you", "merci"),
    ("good night", "bonne nuit"),
    ("i am happy", "je suis heureux"),
    ("i am sad", "je suis triste"),
    ("how are you", "comment allez vous"),
    ("see you", "a bientot")
]

In [3]:
# 2. Create vocabulary
src_words = sorted(set(" ".join(x for x, y in pairs).split()))
tgt_words = sorted(set(" ".join(y for x, y in pairs).split()))

src_vocab = {w: i+1 for i, w in enumerate(src_words)}
tgt_vocab = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2}
tgt_vocab.update({w: i+3 for i, w in enumerate(tgt_words)})

# 3. Convert sentences into sequences
enc = [[src_vocab[w] for w in x.split()] for x, y in pairs]
dec_in = [[1] + [tgt_vocab[w] for w in y.split()] for x, y in pairs]
dec_out = [[tgt_vocab[w] for w in y.split()] + [2] for x, y in pairs]

max_src = max(map(len, enc))
max_tgt = max(map(len, dec_in))

enc = pad_sequences(enc, maxlen=max_src, padding="post")
dec_in = pad_sequences(dec_in, maxlen=max_tgt, padding="post")
dec_out = pad_sequences(dec_out, maxlen=max_tgt, padding="post")

In [4]:
# 4. Encoder
enc_input = Input(shape=(max_src,))
enc_embed = Embedding(len(src_vocab)+1, 32)(enc_input)
enc_out, h, c = LSTM(64, return_sequences=True,
                     return_state=True)(enc_embed)

# 5. Decoder
dec_input = Input(shape=(max_tgt,))
dec_embed = Embedding(len(tgt_vocab), 32)(dec_input)
dec_out_seq = LSTM(64, return_sequences=True)(
    dec_embed, initial_state=[h, c])

# 6. Attention mechanism
context = AdditiveAttention()([dec_out_seq, enc_out])
combined = Concatenate()([dec_out_seq, context])
output = Dense(len(tgt_vocab), activation="softmax")(combined)

In [5]:

# 7. Build and train model
model = Model([enc_input, dec_input], output)
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

model.fit([enc, dec_in], dec_out[..., None],
          epochs=300, verbose=0)

In [6]:
# 8. Evaluate model
loss, accuracy = model.evaluate(
    [enc, dec_in], dec_out[..., None], verbose=0
)
print("Attention Model Accuracy:", round(accuracy*100, 2), "%")

# 9. Display predictions
reverse_vocab = {i: w for w, i in tgt_vocab.items()}

for i, (english, french) in enumerate(pairs[:4]):
    pred = model.predict(
        [enc[i:i+1], dec_in[i:i+1]], verbose=0
    )
    ids = np.argmax(pred[0], axis=1)
    words = [reverse_vocab[x] for x in ids if x > 2]
    print(english, "->", " ".join(words))


Attention Model Accuracy: 100.0 %
hello -> bonjour
good morning -> bonjour
thank you -> merci
good night -> bonne nuit
